In [6]:
import nltk
import heapq
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from transformers import pipeline

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\DC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\DC\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [7]:
text = """Artificial Intelligence is transforming industries.
It enables machines to learn from data.
Machine learning is a subset of AI.
Deep learning further enhances capabilities.
AI is widely used in healthcare, finance, and automation."""

In [8]:
def preprocess(text):
    stop_words = set(stopwords.words('english'))
    words = word_tokenize(text.lower())
    filtered_words = [w for w in words if w.isalnum() and w not in stop_words]
    return filtered_words

In [9]:
def extractive_summary(text, num_sentences=3):
    sentences = sent_tokenize(text)
    words = preprocess(text)

    freq = {}
    for word in words:
        freq[word] = freq.get(word, 0) + 1

    max_freq = max(freq.values())
    for word in freq:
        freq[word] /= max_freq

    scores = {}
    for sent in sentences:
        for word in word_tokenize(sent.lower()):
            if word in freq:
                scores[sent] = scores.get(sent, 0) + freq[word]

    summary_sentences = heapq.nlargest(num_sentences, scores, key=scores.get)
    return ' '.join(summary_sentences)

In [10]:
print("Extractive Summary:\n")
print(extractive_summary(text))

Extractive Summary:

AI is widely used in healthcare, finance, and automation. Machine learning is a subset of AI. Deep learning further enhances capabilities.


In [13]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def abstractive_summary(text):
    inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True)
    summary_ids = model.generate(inputs["input_ids"], max_length=50, min_length=20)
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 511/511 [00:00<00:00, 4276.69it/s]


In [14]:
print("Abstractive Summary:\n")
print(abstractive_summary(text))

Abstractive Summary:

Artificial Intelligence is transforming industries. It enables machines to learn from data. It is widely used in healthcare, finance, and automation.


In [15]:
print("Original Text:\n", text)

print("\nExtractive Summary:\n", extractive_summary(text))

print("\nAbstractive Summary:\n", abstractive_summary(text))

Original Text:
 Artificial Intelligence is transforming industries.
It enables machines to learn from data.
Machine learning is a subset of AI.
Deep learning further enhances capabilities.
AI is widely used in healthcare, finance, and automation.

Extractive Summary:
 AI is widely used in healthcare, finance, and automation. Machine learning is a subset of AI. Deep learning further enhances capabilities.

Abstractive Summary:
 Artificial Intelligence is transforming industries. It enables machines to learn from data. It is widely used in healthcare, finance, and automation.


In [16]:
original_length = len(text.split())
summary_length = len(extractive_summary(text).split())

print("Original Length:", original_length)
print("Summary Length:", summary_length)

Original Length: 33
Summary Length: 21
